# Compare all five methods

Issue: #29

## Question

How do BM25, co-visitation, LightGBM, Two-Tower and DCN-V2 + MMoE compare on the same queries, with the same metrics and rules, overall and inside slices?

## Data used

Per-session results already saved by the earlier notebooks (all scored on the same 56,544 evaluation queries with the shared rules of #17): `data/results/{bm25,covis,lgbm,two_tower,dcn_mmoe}.parquet`, plus `popularity.parquet` as a reference. The frozen benchmark of #16 (`benchmark_v1.parquet`), the queries (#10) and training-window counts for popularity buckets.

Output: `data/results/comparison.json` (the final table).

**Two kinds of method.** *Retrievers* return their own top list from the whole category: BM25, co-visitation (with popularity fill, as in #18) and Two-Tower. *Rerankers* re-order the shared candidate pool of 200 (covis + popularity): LightGBM lambdarank and DCN-V2 + MMoE. Rerankers can never return an item outside the pool, and their recall is limited by it. All rules are the same: rank inside the query category, no prefix items, and the same 56,544 queries (queries with no relevant label are dropped for everyone).

**Known limits, stated once.** The query category comes from the held-out click (a leak, #10). Categories and BM25 documents are synthetic. The results say how the methods compare on this data, not how they would do in a real shop.

In [ ]:
import json
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bootstrap import bootstrap_ci, paired_bootstrap_diff
from slicing import length_bucket, popularity_buckets, sliced_metrics
from split import WINDOWS, day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
R = DATA / "results"
D14 = day_to_ms(WINDOWS["w0 initial (weeks 1-2)"]["train_end_day"])
METHODS = {
    "BM25 (synthetic text)": ("bm25", "retriever"),
    "Co-visitation + fill": ("covis", "retriever"),
    "Two-Tower (logQ)": ("two_tower", "retriever"),
    "LightGBM lambdarank": ("lgbm", "reranker"),
    "DCN-V2 + MMoE": ("dcn_mmoe", "reranker"),
}
METRICS = ["ndcg10", "recall20", "recall100"]
frames = {name: pl.read_parquet(R / f"{f}.parquet") for name, (f, _) in METHODS.items()}
pop_ref = pl.read_parquet(R / "popularity.parquet")
sessions = frames["Co-visitation + fill"]["session"]
for name, df in {**frames, "popularity": pop_ref}.items():
    assert df["session"].to_list() == sessions.to_list(), f"{name} was not scored on the same sessions in the same order"
print(f"all {len(METHODS) + 1} methods were scored on the same {sessions.len():,} sessions")

## Table 1: the full evaluation window

In [ ]:
def ci_table(sel=None):
    rows = []
    for name, (f, kind) in METHODS.items():
        df = frames[name] if sel is None else frames[name].filter(pl.col("session").is_in(sel))
        row = {"method": name, "kind": kind, "n": df.height}
        for m in METRICS:
            mean, lo, hi = bootstrap_ci(df[m].to_numpy())
            row[m] = round(mean, 4); row[m + "_ci"] = f"[{lo:.4f}, {hi:.4f}]"
        rows.append(row)
    return pl.DataFrame(rows)


full = ci_table()
with pl.Config(tbl_width_chars=200, tbl_cols=12):
    print(full)

diff_rows = []
ref = frames["Co-visitation + fill"]
for name in METHODS:
    if name == "Co-visitation + fill":
        continue
    j = frames[name].join(ref.select("session", pl.col("ndcg10").alias("ref")), on="session")
    m, lo, hi = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["ref"].to_numpy())
    diff_rows.append({"method": name, "ndcg10_minus_covis": round(m, 4), "ci": f"[{lo:+.4f}, {hi:+.4f}]", "verdict": "better" if lo > 0 else ("worse" if hi < 0 else "no clear difference")})
diffs = pl.DataFrame(diff_rows)
print(diffs)

## Table 2: the frozen benchmark set (#16)

The benchmark has 300 pairs; those whose query has no relevant label under the shared rules are dropped for every method, so fewer remain. With so few queries the intervals are wide.

In [ ]:
bench = pl.read_parquet(DATA / "eval" / "benchmark_v1.parquet")
bench_sessions = set(bench["session"].to_list()) & set(sessions.to_list())
print(f"benchmark pairs: {bench.height}, of which scored (relevant label exists): {len(bench_sessions)}")
bench_table = ci_table(list(bench_sessions))
with pl.Config(tbl_width_chars=200, tbl_cols=12):
    print(bench_table)
bdiff = []
for name in METHODS:
    if name == "Co-visitation + fill":
        continue
    a = frames[name].filter(pl.col("session").is_in(list(bench_sessions))).sort("session")
    b = ref.filter(pl.col("session").is_in(list(bench_sessions))).sort("session")
    m, lo, hi = paired_bootstrap_diff(a["ndcg10"].to_numpy(), b["ndcg10"].to_numpy())
    bdiff.append({"method": name, "ndcg10_minus_covis": round(m, 4), "ci": f"[{lo:+.4f}, {hi:+.4f}]"})
print(pl.DataFrame(bdiff))

## Table 3: slices

Slices by the popularity of the held-out target item (head / torso / tail from training-window counts before day 14) and by prefix length. Values are mean NDCG@10 with bootstrap intervals; slices with fewer than 100 sessions are flagged.

In [ ]:
queries = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
counts = dict(events.filter(pl.col("ts") < D14).group_by("aid").len().collect().iter_rows())
bucket_of = popularity_buckets(counts)
meta = queries.select("session", pl.col("query_category").alias("category"), pl.col("label_aid").list.first().alias("target")).join(
    frames["Co-visitation + fill"].select("session", "prefix_len"), on="session")
meta = meta.with_columns(
    pl.col("target").map_elements(lambda a: bucket_of.get(int(a), "tail"), return_dtype=pl.Utf8).alias("popularity_bucket"),
    pl.col("prefix_len").map_elements(length_bucket, return_dtype=pl.Utf8).alias("length_bucket"),
)
wide = meta
for name, df in frames.items():
    wide = wide.join(df.select("session", pl.col("ndcg10").alias(name)), on="session", how="inner")
print(wide.height, "sessions in the sliced table")


def slice_table(col):
    rows = []
    for name in METHODS:
        t = sliced_metrics(wide.select(col, name), col, name, n_boot=300)
        for r in t.iter_rows(named=True):
            rows.append({"method": name, col: r["slice"], "n": r["n"], "ndcg10": round(r["metric"], 4), "ci": f"[{r['low']:.4f}, {r['high']:.4f}]", "flag": r["flag"]})
    return pl.DataFrame(rows)


with pl.Config(tbl_rows=30, tbl_width_chars=160):
    pop_slices = slice_table("popularity_bucket")
    print(pop_slices.pivot(on="method", index=["popularity_bucket", "n"], values="ndcg10").sort("popularity_bucket"))
    len_slices = slice_table("length_bucket")
    print(len_slices.pivot(on="method", index=["length_bucket", "n"], values="ndcg10").sort("length_bucket"))

In [ ]:
# category slices: in how many categories does each method beat co-visitation?
cat_mean = wide.group_by("category").agg(pl.len().alias("n"), *[pl.col(m).mean().alias(m) for m in METHODS])
big = cat_mean.filter(pl.col("n") >= 100)
print(f"{big.height} categories with at least 100 sessions (of {cat_mean.height})")
rows = []
for name in METHODS:
    if name == "Co-visitation + fill":
        continue
    better = int((big[name] > big["Co-visitation + fill"]).sum())
    rows.append({"method": name, "categories_better_than_covis": better, "of": big.height})
print(pl.DataFrame(rows))
best = big.with_columns(*[(pl.col(m) - pl.col("Co-visitation + fill")).alias(m + "_gain") for m in ("LightGBM lambdarank",)]).sort("LightGBM lambdarank_gain", descending=True)
print("categories where LightGBM gains most over co-visitation:", best.select("category", "n", "LightGBM lambdarank_gain").head(3).rows())
print("categories where LightGBM loses most:", best.select("category", "n", "LightGBM lambdarank_gain").tail(3).rows())

In [ ]:
# export
out = {
    "note": "Same 56,544 evaluation queries and shared rules for every method. Category from the held-out click (known leak). Categories and BM25 text are synthetic.",
    "full_window": full.to_dicts(),
    "paired_vs_covis": diffs.to_dicts(),
    "benchmark_v1": {"pairs": bench.height, "scored": len(bench_sessions), "table": bench_table.to_dicts(), "paired_vs_covis": bdiff},
    "slices_popularity": pop_slices.to_dicts(),
    "slices_prefix_length": len_slices.to_dicts(),
}
(R / "comparison.json").write_text(json.dumps(out, indent=2, default=str))
assert (R / "comparison.json").exists()
print("saved", R / "comparison.json")

## What surprised me

**Summary (NDCG@10, 56,544 queries, mean [95% CI]):** LightGBM 0.2153 [0.2127, 0.2180] > co-visitation 0.2144 [0.2118, 0.2171] > DCN-V2 + MMoE 0.2069 [0.2043, 0.2096] >> Two-Tower 0.0459 >> BM25 0.0007. Paired against co-visitation: LightGBM +0.0009 [+0.0003, +0.0015] (better), DCN-V2 -0.0075 [-0.0087, -0.0064] (worse), Two-Tower -0.1685 (worse), BM25 -0.2138 (worse).

1. **Almost nothing beats counting.** Co-visitation, which has no training beyond counting pairs, is 0.2144, and the best learned model gains only +0.0009. Two learned rerankers built on top of co-visitation's own candidates and signals, and a neural retriever, add very little or lose. The strongest signal in this data is which items were seen together in a session, and counting captures most of it.
2. **The benchmark set cannot tell the top three apart.** On the 219 benchmark pairs that have a relevant label (of 300), the order is different from the full window (co-visitation 0.2313, LightGBM 0.2238, DCN-V2 0.2164), and no interval separates the three (LightGBM minus co-visitation -0.0075 [-0.0169, +0.0021]); the intervals are about +-0.04 wide. The frozen set is fine for the judge, where 219 pairs is what we can afford, but it cannot rank close methods. Only the full window can.
3. **DCN-V2 is better than co-visitation on mid-popularity items and worse on the head.** Popularity slice (NDCG@10): head (30,836 sessions) co-visitation 0.257, LightGBM 0.257, DCN-V2 0.238; torso (18,963) co-visitation 0.179, LightGBM 0.182, DCN-V2 **0.192**; tail (6,745) co-visitation 0.118, LightGBM 0.117, DCN-V2 0.108. So the deep model gives up head accuracy for a gain on torso items, but the net result is lower. It beats co-visitation in only 16 of 80 categories, LightGBM in 47.
4. **Nobody fixes the tail.** Every method does best when the target is a popular item (0.26 on head) and worst on tail items (0.12), and the rerankers do not reduce that gap. Behaviour-based labels favour popular items, and so do the models.
5. **Two-Tower is popularity in disguise.** It scores 0.082 on head targets and about 0.001 to 0.004 on torso and tail, so it recommends popular items in the category and has learned little about the session.
6. **Short prefixes are easiest.** A one-event prefix gives the highest NDCG@10 (co-visitation 0.234) and a prefix of 11 or more events the lowest (0.152): a single recent item points sharply to its neighbours, while a long session spreads across many.
7. **The methods differ in kind, not only in score:** BM25, co-visitation and Two-Tower are retrievers (they return their own lists); LightGBM and DCN-V2 only re-order the shared pool of 200, which limits them to 0.61 recall@200 and a best possible NDCG@10 of 0.62.

## What I learned

To be written by the owner of the project.